# Stage 1: Data Preparation (Run on Local M4 Mac)

This notebook contains **zero** PyTorch logic. Its sole purpose is to utilize all of your M4 CPU cores to generate the raw `np.memmap` image database and the pandas metadata. 

**Instructions:**
1. Run this notebook completely on your local machine.
2. It will output `raw_n5.dat`, `raw_n20.dat`, `raw_n60.dat` and their `.parquet` metadata files into the `./kaggle_dataset` directory.
3. Zip the `./kaggle_dataset` directory.
4. Upload it to Kaggle as a new Dataset (e.g., named `trend-images`).


In [1]:
import os, multiprocessing as mp
try: mp.set_start_method('fork', force=True)
except RuntimeError: pass
import concurrent.futures as cf
import numpy as np, pandas as pd
from PIL import Image, ImageDraw
from tqdm.auto import tqdm
import warnings; warnings.filterwarnings('ignore')

CPU = mp.cpu_count()
CACHE_DIR = './kaggle_dataset'
os.makedirs(CACHE_DIR, exist_ok=True)
print(f"CPU Cores: {CPU} | Output Dir: {CACHE_DIR}")


CPU Cores: 10 | Output Dir: ./kaggle_dataset


## 1. Load Parquet Data


In [2]:
# Make sure to point this to where your parquet file is locally downloaded
file_name = './market_data_sp500_1997_2023.parquet'
if not os.path.exists(file_name):
    raise FileNotFoundError(f"{file_name} not found! Please download the dataset.")

df_market = pd.read_parquet(file_name)
print(f"Loaded {len(df_market):,} rows of market data.")


Loaded 2,739,387 rows of market data.


## 2. PIL Image Generator


In [3]:
def _height(n_days): return 32 if n_days==5 else (64 if n_days==20 else 96)

def generate_image_uint8(window_df, n_days):
    height = _height(n_days); width = n_days*3
    img = Image.new('L', (width, height), color=0)
    draw = ImageDraw.Draw(img)
    if len(window_df)==0: return np.zeros((1,height,width), dtype=np.uint8)
    first_close = window_df['Close'].iloc[0]
    if first_close==0 or np.isnan(first_close): return np.zeros((1,height,width), dtype=np.uint8)
    prices  = window_df[['Open','High','Low','Close']].values / first_close
    ma      = window_df['Close'].rolling(window=n_days, min_periods=1).mean().values / first_close
    volumes = window_df['Volume'].values
    max_p, min_p = np.nanmax(prices), np.nanmin(prices); max_v = np.nanmax(volumes)
    if max_p==min_p: max_p = min_p + 1e-5
    if max_v==0:     max_v = 1e-5
    price_h = int(height*0.8); vol_h = height - price_h
    def p2y(p): return int(np.clip(price_h-1-(p-min_p)/(max_p-min_p)*(price_h-1),0,price_h-1))
    def v2y(v): return int(np.clip(height-1-(v/max_v)*(vol_h-1),price_h,height-1))
    ma_pts=[]
    for i in range(len(window_df)):
        x=i*3+1
        oy,hy,ly,cy = p2y(prices[i,0]),p2y(prices[i,1]),p2y(prices[i,2]),p2y(prices[i,3])
        draw.line([(x,hy),(x,ly)],fill=255,width=1)
        draw.line([(x-1,oy),(x,oy)],fill=255,width=1)
        draw.line([(x,cy),(x+1,cy)],fill=255,width=1)
        draw.line([(x,height-1),(x,v2y(volumes[i]))],fill=255,width=1)
        ma_pts.append((x,p2y(ma[i])))
    if len(ma_pts)>1: draw.line(ma_pts,fill=255,width=1)
    return np.array(img, dtype=np.uint8)


## 3. Pure Memmap Data Store (2-Pass Generation)


In [4]:
REQUIRED = ['Open','High','Low','Close','Volume','MOM','STR','WSTR','TREND']

def _index_ticker(args):
    ticker, t_df, n_days = args
    t_df = t_df.sort_values('Date').reset_index(drop=True)
    nan_count = t_df[REQUIRED].isna().rolling(window=n_days).sum().sum(axis=1)
    valid = np.where(nan_count.values == 0)[0]
    
    samples = []
    for idx in valid:
        samples.append({
            'ticker': ticker, 'end_idx': int(idx), 'end_date': str(t_df['Date'].iloc[idx].date()),
            'MOM': float(t_df['MOM'].iloc[idx]), 'STR': float(t_df['STR'].iloc[idx]),
            'WSTR': float(t_df['WSTR'].iloc[idx]), 'TREND': float(t_df['TREND'].iloc[idx])
        })
    return ticker, t_df, samples

_G = {}
def _render_memmap_chunk(rng):
    lo, hi = rng
    mm = np.memmap(_G['dat_path'], dtype=np.uint8, mode='r+', shape=_G['shape'])
    nd = _G['n_days']
    for i in range(lo, hi):
        s = _G['samples'][i]; tdf = _G['grouped'][s['ticker']]
        idx = s['end_idx']
        mm[i, 0] = generate_image_uint8(tdf.iloc[idx-nd+1:idx+1], nd)
    mm.flush()
    return hi - lo

def build_memmap_store(df, n_days, dat_path, meta_path):
    H, W = _height(n_days), n_days*3
    
    if os.path.exists(dat_path) and os.path.exists(meta_path):
        print(f"  [CACHE HIT] {dat_path} already exists. Skipping.")
        return
        
    print(f"  [CACHE MISS] Generating new images for N={n_days}...")
    
    args = [(t, df[df['Ticker']==t].copy(), n_days) for t in df['Ticker'].unique()]
    grouped, samples = {}, []
    
    with cf.ProcessPoolExecutor(max_workers=CPU) as ex:
        for tk, tdf, s in tqdm(ex.map(_index_ticker, args), total=len(args), desc=f"Index N={n_days}"):
            grouped[tk] = tdf
            samples.extend(s)
            
    N = len(samples)
    if N == 0: return
    
    np.memmap(dat_path, dtype=np.uint8, mode='w+', shape=(N, 1, H, W)).flush()
    pd.DataFrame(samples).to_parquet(meta_path)
    
    _G.update(dat_path=dat_path, shape=(N, 1, H, W), n_days=n_days, grouped=grouped, samples=samples)
    
    chunk = max(1, N // (CPU * 4))
    ranges = [(i, min(i + chunk, N)) for i in range(0, N, chunk)]
    
    with cf.ProcessPoolExecutor(max_workers=CPU) as ex:
        list(tqdm(ex.map(_render_memmap_chunk, ranges), total=len(ranges), desc=f"Render N={n_days}"))
        
    mb = (N * H * W) / 1e6
    print(f"  built memmap {dat_path}: {N:,} images (~{mb:,.0f} MB)\n")


## 4. Run Generation


In [5]:
IMAGE_SIZES = [5, 20, 60]

for n_days in IMAGE_SIZES:
    dat_path = os.path.join(CACHE_DIR, f'raw_n{n_days}.dat')
    meta_path = os.path.join(CACHE_DIR, f'meta_n{n_days}.parquet')
    build_memmap_store(df_market, n_days, dat_path, meta_path)

print("Data Preparation Complete! Zip the './kaggle_dataset' folder and upload it to Kaggle.")


  [CACHE MISS] Generating new images for N=5...


Index N=5:   0%|          | 0/495 [00:00<?, ?it/s]

Render N=5:   0%|          | 0/41 [00:00<?, ?it/s]

  built memmap ./kaggle_dataset/raw_n5.dat: 2,739,387 images (~1,315 MB)

  [CACHE MISS] Generating new images for N=20...


Index N=20:   0%|          | 0/495 [00:00<?, ?it/s]

Render N=20:   0%|          | 0/41 [00:00<?, ?it/s]

  built memmap ./kaggle_dataset/raw_n20.dat: 2,739,387 images (~10,519 MB)

  [CACHE MISS] Generating new images for N=60...


Index N=60:   0%|          | 0/495 [00:00<?, ?it/s]

Render N=60:   0%|          | 0/41 [00:00<?, ?it/s]

  built memmap ./kaggle_dataset/raw_n60.dat: 2,739,387 images (~47,337 MB)

Data Preparation Complete! Zip the './kaggle_dataset' folder and upload it to Kaggle.
